# cAPTure: Stage-2 training preflight

This notebook freezes and audits the chronological inner-checkpoint split before any model training. It tests the declared 20%, 25%, and 30% validation tails using development scenarios only, refuses to split attack-step iterations, verifies graph/packet alignment, and runs inference-only synthetic model-interface checks.

It does not optimize model parameters, select score thresholds, or access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

## 1. Mount Drive and prepare the environment

Push the current branch before opening Colab. Use a fresh runtime so imported modules cannot come from an older commit.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_PREFLIGHT_WORK_ROOT = Path("/content/capture_graph_preflight_work")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture-mlp.txt",
    "code/python/utils/capture_graph_dataset.py",
    "code/python/utils/capture_graph_training_preflight.py",
    "configs/capture_graph_stage2_input_v2.yaml",
    "configs/capture_graph_training_preflight_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_training_preflight import (
    load_completed_training_preflight,
    run_capture_graph_training_preflight,
    save_training_preflight_decision,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Training-preflight environment is ready.")

Mounted at /content/drive
Repository commit: 4befe1db0cbb3339cdb84a555727161f95c74598
Training-preflight environment is ready.


## 2. Bind the reviewed immutable inputs

The preflight contract names the exact prepared, graph-materialization, and graph-input-audit runs. A different run ID or checksum is rejected.

In [2]:
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
GRAPH_INPUT_AUDIT_RUN_ID = "20260928T193308_017682Z_graph_input_full"

PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
GRAPH_INPUT_AUDIT_DIR = (
    DRIVE_ROOT / "graph_input_audit_runs" / GRAPH_INPUT_AUDIT_RUN_ID
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
PREFLIGHT_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_preflight_v1.yaml"
)
for path in (
    PREPARED_RUN_DIR, GRAPH_MATERIALIZATION_DIR, GRAPH_INPUT_AUDIT_DIR,
    GRAPH_INPUT_CONTRACT_PATH, PREFLIGHT_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required preflight input is missing: {path}")

print("Prepared packets:", PREPARED_RUN_DIR)
print("Graph materialization:", GRAPH_MATERIALIZATION_DIR)
print("Graph-input audit:", GRAPH_INPUT_AUDIT_DIR)
print("Preflight contract:", PREFLIGHT_CONFIG_PATH)

Prepared packets: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Graph materialization: /content/drive/MyDrive/capture_gate0/graph_materialization_runs/20260928T175821_521784Z_graph_materialization_full_dev
Graph-input audit: /content/drive/MyDrive/capture_gate0/graph_input_audit_runs/20260928T193308_017682Z_graph_input_full
Preflight contract: /content/temporalgnn-nids/configs/capture_graph_training_preflight_v1.yaml


## 3. Stage and verify the graph collection

The existing FULL_DEV materialization is copied to local Colab storage and every artifact checksum is verified. No graph is rebuilt.

In [3]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
print("Verified local graph root:", collection.root)
print("All graph checksums verified:",
      collection.artifact_checksums_verified)

Staged and verified graph materialization: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
Verified local graph root: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
All graph checksums verified: True


## 4. Run or resume the training preflight

This scans only prepared development metadata, evaluates the declared chronological suffixes, verifies their graph positions, and performs inference-only synthetic forward/reset checks. It may take several minutes because the prepared artifacts are checksum-verified and staged one scenario at a time.

In [4]:
PREFLIGHT_RUN_ID = "20260928T201318_972670Z_graph_training_preflight"  # Paste an existing completed ID to reuse it.
if PREFLIGHT_RUN_ID is None:
    PREFLIGHT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_training_preflight"
    )
PREFLIGHT_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_training_preflight_runs" / PREFLIGHT_RUN_ID
)
print("Training-preflight run ID:", PREFLIGHT_RUN_ID)
print("Training-preflight output:", PREFLIGHT_OUTPUT_DIR)

if PREFLIGHT_OUTPUT_DIR.exists():
    preflight_report = load_completed_training_preflight(
        PREFLIGHT_OUTPUT_DIR,
        collection=collection,
        config_path=PREFLIGHT_CONFIG_PATH,
    )
    print("Reused the completed immutable training preflight.")
else:
    preflight_report = run_capture_graph_training_preflight(
        collection=collection,
        prepared_run_dir=PREPARED_RUN_DIR,
        graph_input_audit_dir=GRAPH_INPUT_AUDIT_DIR,
        config_path=PREFLIGHT_CONFIG_PATH,
        output_dir=PREFLIGHT_OUTPUT_DIR,
        local_work_root=LOCAL_PREFLIGHT_WORK_ROOT,
    )
    print("Saved the immutable training preflight.")

Training-preflight run ID: 20260928T201318_972670Z_graph_training_preflight
Training-preflight output: /content/drive/MyDrive/capture_gate0/graph_training_preflight_runs/20260928T201318_972670Z_graph_training_preflight
Reused the completed immutable training preflight.


## 5. Review split candidates and the selected boundary

A candidate is globally valid only when every development scenario passes. Inspect all rejection reasons and verify that the selected split has both packet labels, benign-only and attack-containing windows, complete iterations, and multiple steps on both sides.

In [5]:
candidate_table = pd.read_csv(PREFLIGHT_OUTPUT_DIR / "split_candidates.csv")
try:
    split_table = pd.read_csv(PREFLIGHT_OUTPUT_DIR / "split_summary.csv")
except pd.errors.EmptyDataError:
    # Older blocked preflight runs wrote this artifact without a header.
    split_table = pd.DataFrame()
display(candidate_table)
print("Preflight status:", preflight_report["status"])
print("Selected validation tail fraction:",
      preflight_report["selected_validation_tail_fraction"])
print("Candidate valid for every scenario:", json.dumps(
    preflight_report["candidate_is_valid_for_every_scenario"], indent=2
))
print("Fold totals:", json.dumps(preflight_report["fold_totals"], indent=2))
if split_table.empty:
    print("No split was selected. Review the rejection reasons above.")
else:
    display(split_table)

,scenario,candidate_validation_tail_fraction,target_boundary_window_index,maximum_boundary_shift_windows,boundary_window_index,boundary_shift_windows,realized_validation_tail_fraction,admissible,rejection_reasons,split_iterations,...,inner_train.distinct_attack_steps,inner_validation.wall_clock_windows,inner_validation.nonempty_windows,inner_validation.benign_only_windows,inner_validation.attack_containing_windows,inner_validation.packets,inner_validation.normal_packets,inner_validation.attack_packets,inner_validation.complete_attack_iterations,inner_validation.distinct_attack_steps
0,train_empty_conn,0.20,7488,188,7488,0,0.200000,False,"[""inner_validation.attack_containing_windows=0...",[],...,9,1872,1872,1872,0,139599,139599,0,0,0
1,train_qos_mid,0.20,7488,188,7488,0,0.200000,False,"[""inner_validation.attack_containing_windows=0...",[],...,9,1872,1872,1872,0,139599,139599,0,0,0
2,train_dollar_char,0.20,7488,188,7501,13,0.198611,True,[],[],...,8,1859,1616,1490,126,111182,69302,41880,12,2
3,train_slash_char,0.20,7488,188,7492,4,0.199573,True,[],[],...,9,1868,1626,1503,123,304949,71345,233604,19,2
4,train_sub_exf,0.20,7488,188,7488,0,0.200000,False,"[""inner_validation.attack_containing_windows=9...",[],...,7,1872,1629,1539,90,78646,72487,6159,14,4
5,train_empty_conn,0.25,7020,188,7078,58,0.243803,False,"[""inner_validation.attack_containing_windows=0...",[],...,9,2282,2282,2282,0,171181,171181,0,0,0
6,train_qos_mid,0.25,7020,188,7000,-20,0.252137,False,"[""inner_validation.complete_attack_iterations=...",[],...,9,2360,2360,2173,187,191168,177830,13338,4,1
7,train_dollar_char,0.25,7020,188,7019,-1,0.250107,True,[],[],...,8,2341,2098,1725,373,266860,171926,94934,27,3
8,train_slash_char,0.25,7020,188,7020,0,0.250000,True,[],[],...,9,2340,2098,1807,291,731860,171794,560066,43,2
9,train_sub_exf,0.25,7020,188,7026,6,0.249359,True,[],[],...,7,2334,2091,1842,249,187492,170502,16990,35,4


Preflight status: blocked
Selected validation tail fraction: None
Candidate valid for every scenario: {
  "0.2": false,
  "0.25": false,
  "0.3": false
}
Fold totals: {
  "A": {
    "inner_train_edges": 0,
    "inner_validation_edges": 0,
    "inner_train_graphs": 0,
    "inner_validation_graphs": 0
  },
  "B": {
    "inner_train_edges": 0,
    "inner_validation_edges": 0,
    "inner_train_graphs": 0,
    "inner_validation_graphs": 0
  }
}
No split was selected. Review the rejection reasons above.


## 6. Review the synthetic model-interface contract

Every variant must produce one finite logit per synthetic edge without mutating shared inputs. Temporal variants must create and then completely reset their state. No backward pass or optimizer is used.

In [6]:
model_contract = preflight_report["model_contract"]
model_rows = [
    {"variant": name, **result}
    for name, result in model_contract["variants"].items()
]
display(pd.DataFrame(model_rows))
print("Shared inputs unchanged:", model_contract["shared_inputs_unchanged"])
print("Optimization performed:", model_contract["optimization_performed"])
print("Held-out scenarios accessed:",
      preflight_report["held_out_scenarios_accessed"])
print("Model training performed:",
      preflight_report["model_training_performed"])

,variant,status,temporal,temporal_state_created,temporal_reset_verified,logits,parameters
0,edge_mlp,passed,False,False,False,4,4545
1,edge_gru,passed,True,True,True,4,16257
2,static_gnn,passed,False,False,False,4,24097
3,st_gnn,passed,True,True,True,4,30433
4,st_gnn_without_gat,passed,True,True,True,4,13921
5,st_gnn_without_direct_edge_attr,passed,True,True,True,4,27137


Shared inputs unchanged: True
Optimization performed: False
Held-out scenarios accessed: False
Model training performed: False


## 7. Record the manual preflight decision

Approval freezes only the inner split and authorizes creation of the full training configuration and runner. It does not authorize training yet. Do not approve a blocked preflight.

In [ ]:
APPROVE_TRAINING_PREFLIGHT = False
PREFLIGHT_REVIEW_NOTE = "Unapproved"
DECISION_PATH = PREFLIGHT_OUTPUT_DIR / "training_preflight_decision.json"

if DECISION_PATH.exists():
    decision = json.loads(DECISION_PATH.read_text(encoding="utf-8"))
    print("Existing preflight decision:", json.dumps(decision, indent=2))
else:
    if not APPROVE_TRAINING_PREFLIGHT:
        raise RuntimeError(
            "Review the preflight, then explicitly approve or retain the run as unapproved."
        )
    decision = save_training_preflight_decision(
        PREFLIGHT_OUTPUT_DIR,
        approved=True,
        review_note=PREFLIGHT_REVIEW_NOTE,
    )
    print("Saved preflight decision:", json.dumps(decision, indent=2))